In [2]:
import pandas as pd
import numpy as np



In [4]:
dtypes = {
    "event_type": "category",
    "product_id": "int64",
    "category_id": "int64",
    "brand": "category",
    "price": "float64",
    "user_id": "int64",
    "user_session": "string",
    "brand_imputed": "bool",
    "brand_fallback": "bool",
}

df = pd.read_csv("../data/new_clean_events.csv", dtype=dtypes)

In [6]:
# Convert event_time using the exact format saved in the CSV
df["event_time"] = pd.to_datetime(
    df["event_time"], format="%Y-%m-%d %H:%M:%S UTC", utc=True
)

# Keep the original row order as a tie-breaker for events with the same timestamp
df["orig_order"] = np.arange(len(df))

# Sort by user and time (orig_order keeps the order stable for equal timestamps)
df = df.sort_values(["user_id", "event_time", "orig_order"]).reset_index(drop=True)

print("Shape:", df.shape)
print("Unique users:", df["user_id"].nunique())
print(df.dtypes)
print(df[["user_id", "event_time", "event_type", "user_session"]].head(10))

Shape: (11835068, 11)
Unique users: 1011684
event_time        datetime64[us, UTC]
event_type                   category
product_id                      int64
category_id                     int64
brand                        category
price                         float64
user_id                         int64
user_session                   string
brand_imputed                    bool
brand_fallback                   bool
orig_order                      int64
dtype: object
   user_id                event_time event_type  \
0  1120748 2019-11-04 15:28:57+00:00       view   
1  1180452 2019-12-28 14:32:56+00:00       view   
2  1458813 2019-11-19 09:56:09+00:00       view   
3  1458813 2019-11-19 11:14:48+00:00       view   
4  1458813 2019-11-20 11:58:06+00:00       view   
5  1458813 2019-11-20 11:58:27+00:00       view   
6  1458813 2019-11-20 11:59:39+00:00       view   
7  1458813 2019-11-20 12:00:07+00:00       view   
8  1458813 2019-11-20 12:02:58+00:00       view   
9  1458813 201

In [7]:
GAP_MINUTES = 30

# Time gap between two consecutive events of the same user
prev_time = df.groupby("user_id")["event_time"].shift(1)
df["gap_min"] = ((df["event_time"] - prev_time).dt.total_seconds() / 60).astype("float32")
del prev_time

# A new session starts at a user's first event or after a gap longer than the cutoff
df["new_session"] = df["gap_min"].isna() | (df["gap_min"] > GAP_MINUTES)

# Global integer session id (data is sorted by user, so ids never mix users)
df["session_id"] = df["new_session"].cumsum().astype("int32")

n_sessions = df["session_id"].nunique()
print("Total sessions:", n_sessions)
print("Average events per session:", round(len(df) / n_sessions, 2))
print("\nGap (minutes) summary:")
print(df["gap_min"].describe())

Total sessions: 1937785
Average events per session: 6.11

Gap (minutes) summary:
count    1.082338e+07
mean     6.685400e+02
std      4.892063e+03
min      0.000000e+00
25%      8.333334e-02
50%      4.000000e-01
75%      1.550000e+00
max      1.309895e+05
Name: gap_min, dtype: float64


In [8]:
# Convert the long session text into integer codes, then drop the text column
df["orig_session_code"] = pd.factorize(df["user_session"])[0].astype("int32")
df = df.drop(columns=["user_session"])

# Original sessions split into several reconstructed sessions
new_per_orig = df.groupby("orig_session_code")["session_id"].nunique()
print("Original sessions:", len(new_per_orig))
print("Split into 2+ sessions:", (new_per_orig > 1).sum(),
      f"({(new_per_orig > 1).mean() * 100:.2f}%)")

# Reconstructed sessions that merge several original sessions
orig_per_new = df.groupby("session_id")["orig_session_code"].nunique()
print("\nReconstructed sessions:", len(orig_per_new))
print("Merging 2+ original sessions:", (orig_per_new > 1).sum(),
      f"({(orig_per_new > 1).mean() * 100:.2f}%)")

# Session duration and size
sess = df.groupby("session_id").agg(
    start=("event_time", "min"),
    end=("event_time", "max"),
    n_events=("event_type", "size"),
)
sess["duration_min"] = (sess["end"] - sess["start"]).dt.total_seconds() / 60
print("\nSession duration (minutes):")
print(sess["duration_min"].describe())
print("\nEvents per session:")
print(sess["n_events"].describe())
print("Sessions with 1 event:", (sess["n_events"] == 1).sum())

Original sessions: 2641724
Split into 2+ sessions: 100900 (3.82%)

Reconstructed sessions: 1937785
Merging 2+ original sessions: 272605 (14.07%)

Session duration (minutes):
count    1.937785e+06
mean     6.110125e+00
std      1.514436e+01
min      0.000000e+00
25%      0.000000e+00
50%      0.000000e+00
75%      4.583333e+00
max      6.519000e+02
Name: duration_min, dtype: float64

Events per session:
count    1.937785e+06
mean     6.107524e+00
std      1.548501e+01
min      1.000000e+00
25%      1.000000e+00
50%      1.000000e+00
75%      4.000000e+00
max      1.885000e+03
Name: n_events, dtype: float64
Sessions with 1 event: 985267


In [9]:
# Count each event type per session
counts = (
    df.groupby(["session_id", "event_type"], observed=True)
    .size()
    .unstack(fill_value=0)
)
counts.columns = ["n_" + str(c) for c in counts.columns]

# Attach the counts to the session table built in Cell 3
sess = sess.join(counts)
print("Session table columns:", sess.columns.tolist())

has_cart = sess["n_cart"] > 0
has_purchase = sess["n_purchase"] > 0
has_remove = sess["n_remove_from_cart"] > 0

print("\nTotal sessions:", len(sess))
print("Sessions with cart:", has_cart.sum())
print("Sessions with purchase:", has_purchase.sum())
print("Sessions with remove_from_cart:", has_remove.sum())

# Cart sessions: what happened in them
cart = sess[has_cart]
print("\nCart sessions:", len(cart))
print("  with purchase:", (cart["n_purchase"] > 0).sum())
print("  with remove:", (cart["n_remove_from_cart"] > 0).sum())
print("  with purchase and remove:",
      ((cart["n_purchase"] > 0) & (cart["n_remove_from_cart"] > 0)).sum())
print("  removes more than cart adds:",
      (cart["n_remove_from_cart"] > cart["n_cart"]).sum())

# Events that belong to a cart started in an earlier session
print("\nPurchase sessions WITHOUT a cart event:", (has_purchase & ~has_cart).sum())
print("Remove sessions WITHOUT a cart event:", (has_remove & ~has_cart).sum())

Session table columns: ['start', 'end', 'n_events', 'duration_min', 'n_cart', 'n_purchase', 'n_remove_from_cart', 'n_view']

Total sessions: 1937785
Sessions with cart: 529865
Sessions with purchase: 94785
Sessions with remove_from_cart: 275879

Cart sessions: 529865
  with purchase: 81398
  with remove: 229361
  with purchase and remove: 59638
  removes more than cart adds: 67096

Purchase sessions WITHOUT a cart event: 13387
Remove sessions WITHOUT a cart event: 46518


In [10]:
# Cart events and purchase events of the same user and product
cart_ev = df.loc[
    df["event_type"] == "cart", ["user_id", "product_id", "event_time", "session_id"]
].sort_values("event_time")

pur_ev = df.loc[
    df["event_type"] == "purchase", ["user_id", "product_id", "event_time", "session_id"]
].rename(columns={"event_time": "purchase_time", "session_id": "purchase_session_id"})
pur_ev = pur_ev.sort_values("purchase_time")

# For each cart event, find the first later purchase of the same user and product
look = pd.merge_asof(
    cart_ev,
    pur_ev,
    left_on="event_time",
    right_on="purchase_time",
    by=["user_id", "product_id"],
    direction="forward",
)

bought = look["purchase_time"].notna()
same_session = bought & (look["purchase_session_id"] == look["session_id"])
later_session = bought & ~same_session
lag_hours = (look["purchase_time"] - look["event_time"]).dt.total_seconds() / 3600

print("Cart events:", len(look))
print("Followed by purchase of same product:", bought.sum(), f"({bought.mean() * 100:.2f}%)")
print("  in the same session:", same_session.sum())
print("  in a LATER session:", later_session.sum())

print("\nLag (hours) for purchases in a later session:")
print(lag_hours[later_session].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.99]))

for label, hours in [("1 hour", 1), ("24 hours", 24), ("7 days", 168), ("30 days", 720)]:
    share = (later_session & (lag_hours <= hours)).sum() / later_session.sum() * 100
    print(f"Later-session purchases within {label}: {share:.1f}%")

# Cart events in the last days of the data cannot be judged fairly
print("\nData ends at:", df["event_time"].max())

Cart events: 3421733
Followed by purchase of same product: 973585 (28.45%)
  in the same session: 557002
  in a LATER session: 416583

Lag (hours) for purchases in a later session:
count    416583.000000
mean        120.897152
std         222.825432
min           0.500278
25%           6.348056
50%          27.069444
75%         123.451528
90%         355.574778
99%        1125.849633
max        2169.496389
dtype: float64
Later-session purchases within 1 hour: 2.7%
Later-session purchases within 24 hours: 46.4%
Later-session purchases within 7 days: 79.6%
Later-session purchases within 30 days: 96.7%

Data ends at: 2019-12-31 23:59:57+00:00


In [11]:
WINDOW_DAYS = 7
window = pd.Timedelta(days=WINDOW_DAYS)
data_end = df["event_time"].max()

# Cart events, keeping the original row id of df
cart_ev = (
    df.loc[df["event_type"] == "cart",
           ["user_id", "product_id", "category_id", "price", "event_time", "session_id"]]
    .reset_index()
    .rename(columns={"index": "row_id"})
    .sort_values("event_time")
)

# First later purchase and first later remove of the same user and product
pur_ev = (
    df.loc[df["event_type"] == "purchase", ["user_id", "product_id", "event_time"]]
    .rename(columns={"event_time": "purchase_time"})
    .sort_values("purchase_time")
)
rem_ev = (
    df.loc[df["event_type"] == "remove_from_cart", ["user_id", "product_id", "event_time"]]
    .rename(columns={"event_time": "remove_time"})
    .sort_values("remove_time")
)

lbl = pd.merge_asof(cart_ev, pur_ev, left_on="event_time", right_on="purchase_time",
                    by=["user_id", "product_id"], direction="forward")
lbl = pd.merge_asof(lbl, rem_ev, left_on="event_time", right_on="remove_time",
                    by=["user_id", "product_id"], direction="forward")

# Keep only events that happen inside the window after the cart event
p = lbl["purchase_time"].where(lbl["purchase_time"] - lbl["event_time"] <= window)
r = lbl["remove_time"].where(lbl["remove_time"] - lbl["event_time"] <= window)

# Whichever happens first decides the outcome
is_cleared = r.notna() & (p.isna() | (r < p))
is_purchased = p.notna() & ~is_cleared

lbl["outcome"] = np.select(
    [is_purchased, is_cleared], ["purchased", "cleared"], default="abandoned"
)

# Abandoned carts near the end of the data did not get a full window
lbl["censored"] = (lbl["outcome"] == "abandoned") & (lbl["event_time"] + window > data_end)

print("Cart events:", len(lbl))
print("\nOutcome counts:")
print(lbl["outcome"].value_counts())
print("\nOutcome share (%):")
print((lbl["outcome"].value_counts(normalize=True) * 100).round(2))
print("\nCensored abandoned carts (last", WINDOW_DAYS, "days):", int(lbl["censored"].sum()))

lbl["month"] = lbl["event_time"].dt.strftime("%Y-%m")
print("\nOutcome by month:")
print(pd.crosstab(lbl["month"], lbl["outcome"]))

Cart events: 3421733

Outcome counts:
outcome
abandoned    1396029
cleared      1288313
purchased     737391
Name: count, dtype: int64

Outcome share (%):
outcome
abandoned    40.80
cleared      37.65
purchased    21.55
Name: proportion, dtype: float64

Censored abandoned carts (last 7 days): 62464

Outcome by month:
outcome  abandoned  cleared  purchased
month                                 
2019-10     560150   421944     231030
2019-11     469251   519692     304986
2019-12     366628   346677     201375


In [12]:
cleared = lbl["outcome"] == "cleared"

# 1. How fast after the cart event was the item removed?
lag_min = (lbl["remove_time"] - lbl["event_time"]).dt.total_seconds() / 60
print("Remove lag (minutes) for cleared carts:")
print(lag_min[cleared].describe(percentiles=[0.25, 0.5, 0.75, 0.9]))
for label, m in [("1 minute", 1), ("5 minutes", 5), ("30 minutes", 30), ("24 hours", 1440)]:
    print(f"Removed within {label}: {(lag_min[cleared] <= m).mean() * 100:.1f}%")

# 2. Was the same product added to the cart more than once by the same user?
lbl["pair_carts"] = lbl.groupby(["user_id", "product_id"])["row_id"].transform("size")
print("\nShare of events where the user added the same product 2+ times (%):")
print(pd.crosstab(lbl["pair_carts"] > 1, lbl["outcome"], normalize="columns").round(3) * 100)

# 3. Cleared carts where the product was still bought within the window
bought_anyway = cleared & lbl["purchase_time"].notna() & (
    lbl["purchase_time"] - lbl["event_time"] <= window
)
print("\nCleared carts that were still purchased within the window:",
      int(bought_anyway.sum()), f"({bought_anyway.sum() / cleared.sum() * 100:.1f}% of cleared)")

# 4. Outcome by price band
lbl["price_band"] = pd.qcut(lbl["price"], 5, duplicates="drop")
print("\nOutcome share by price band (%):")
print((pd.crosstab(lbl["price_band"], lbl["outcome"], normalize="index") * 100).round(1))

Remove lag (minutes) for cleared carts:
count    1.288313e+06
mean     1.068704e+03
std      2.186162e+03
min      0.000000e+00
25%      4.216667e+00
50%      2.266667e+01
75%      8.797333e+02
90%      4.299267e+03
max      1.008000e+04
dtype: float64
Removed within 1 minute: 12.0%
Removed within 5 minutes: 27.3%
Removed within 30 minutes: 53.6%
Removed within 24 hours: 79.8%

Share of events where the user added the same product 2+ times (%):
outcome     abandoned  cleared  purchased
pair_carts                               
False            72.3     61.6       67.3
True             27.7     38.4       32.7

Cleared carts that were still purchased within the window: 151400 (11.8% of cleared)

Outcome share by price band (%):
outcome         abandoned  cleared  purchased
price_band                                   
(0.049, 1.3]         38.2     33.8       28.0
(1.3, 2.52]          40.4     37.7       22.0
(2.52, 4.11]         41.0     39.4       19.6
(4.11, 6.27]         41.3     39.

In [13]:
# 1. Cleared carts that were bought anyway inside the window become "purchased"
p_in = lbl["purchase_time"].where(lbl["purchase_time"] - lbl["event_time"] <= window)
rebought = (lbl["outcome"] == "cleared") & p_in.notna()

lbl["outcome_raw"] = lbl["outcome"]
lbl["outcome"] = lbl["outcome"].where(~rebought, "purchased")
lbl["removed_then_rebought"] = rebought

# 2. Flag quick removes (within 1 minute of the cart event)
remove_lag_min = (lbl["remove_time"] - lbl["event_time"]).dt.total_seconds() / 60
lbl["quick_remove"] = (lbl["outcome_raw"] == "cleared") & (remove_lag_min <= 1)

# 3. Binary target: 1 = not bought (abandoned or cleared), 0 = purchased
lbl["is_abandoned"] = lbl["outcome"].isin(["abandoned", "cleared"]).astype("int8")

# 4. Keep only label columns (lookahead columns are dropped to avoid leakage)
labels = lbl[[
    "row_id", "session_id", "user_id", "product_id", "category_id", "price",
    "event_time", "outcome", "is_abandoned", "censored",
    "quick_remove", "removed_then_rebought",
]].copy()

# Checks
print("Cart events:", len(labels))
print("\nOutcome counts (all):")
print(labels["outcome"].value_counts())

usable = labels[~labels["censored"]]
print("\nUsable rows (not censored):", len(usable))
print("\nOutcome share, usable rows (%):")
print((usable["outcome"].value_counts(normalize=True) * 100).round(2))
print("\nis_abandoned share, usable rows (%):")
print((usable["is_abandoned"].value_counts(normalize=True) * 100).round(2))

print("\nCleared -> rebought:", int(labels["removed_then_rebought"].sum()))
print("Quick removes:", int(labels["quick_remove"].sum()))
print("\nColumns:", labels.columns.tolist())

Cart events: 3421733

Outcome counts (all):
outcome
abandoned    1396029
cleared      1136913
purchased     888791
Name: count, dtype: int64

Usable rows (not censored): 3359269

Outcome share, usable rows (%):
outcome
abandoned    39.70
cleared      33.84
purchased    26.46
Name: proportion, dtype: float64

is_abandoned share, usable rows (%):
is_abandoned
1    73.54
0    26.46
Name: proportion, dtype: float64

Cleared -> rebought: 151400
Quick removes: 154085

Columns: ['row_id', 'session_id', 'user_id', 'product_id', 'category_id', 'price', 'event_time', 'outcome', 'is_abandoned', 'censored', 'quick_remove', 'removed_then_rebought']


In [15]:
df[[
    "user_id", "event_time", "event_type", "product_id", "category_id",
    "brand", "price", "brand_imputed", "brand_fallback", "session_id",
]].to_csv("../data/reconstructed_events.csv", index=False)

print("Saved: ../data/new_reconstructed_events.csv")

Saved: ../data/new_reconstructed_events.csv
